In [ ]:
import requests, json, re
from bs4 import BeautifulSoup

COUNTRY_CODES = {
    'af': 'Afghanistan',
    'al': 'Albanie',
    'dz': 'Algérie',
    'as': 'Samoa américaines',
    'ad': 'Andorre',
    'ao': 'Angola',
    'ag': 'Antigua-et-Barbuda',
    'ar': 'Argentine',
    'am': 'Arménie',
    'aw': 'Aruba',
    'au': 'Australie',
    'at': 'Autriche',
    'az': 'Azerbaïdjan',
    'bs': 'Bahamas',
    'bh': 'Bahreïn',
    'bd': 'Bangladesh',
    'bb': 'Barbade',
    'by': 'Biélorussie',
    'be': 'Belgique',
    'bz': 'Belize',
    'bj': 'Bénin',
    'bm': 'Bermudes',
    'bt': 'Bhoutan',
    'bo': 'Bolivie',
    'ba': 'Bosnie-Herzégovine',
    'bw': 'Botswana',
    'br': 'Brésil',
    'vg': 'Îles Vierges britanniques',
    'bn': 'Brunei',
    'bg': 'Bulgarie',
    'bf': 'Burkina Faso',
    'bi': 'Burundi',
    'kh': 'Cambodge',
    'cm': 'Cameroun',
    'ca': 'Canada',
    'cv': 'Cap-Vert',
    'ky': 'Îles Caïmans',
    'cf': 'République centrafricaine',
    'td': 'Tchad',
    'cl': 'Chili',
    'cn': 'Chine',
    'co': 'Colombie',
    'km': 'Comores',
    'cg': 'Congo',
    'cd': 'République démocratique du Congo',
    'ck': 'Îles Cook',
    'cr': 'Costa Rica',
    'ci': 'Côte d\'Ivoire',
    'hr': 'Croatie',
    'cu': 'Cuba',
    'cy': 'Chypre',
    'cz': 'République tchèque',
    'dk': 'Danemark',
    'dj': 'Djibouti',
    'dm': 'Dominique',
    'do': 'République dominicaine',
    'ec': 'Équateur',
    'eg': 'Égypte',
    'sv': 'El Salvador',
    'gq': 'Guinée équatoriale',
    'er': 'Érythrée',
    'ee': 'Estonie',
    'et': 'Éthiopie',
    'fj': 'Fidji',
    'fi': 'Finlande',
    'fr': 'France',
    'ga': 'Gabon',
    'gm': 'Gambie',
    'ge': 'Géorgie',
    'de': 'Allemagne',
    'gh': 'Ghana',
    'gr': 'Grèce',
    'gd': 'Grenade',
    'gu': 'Guam',
    'gt': 'Guatemala',
    'gn': 'Guinée',
    'gw': 'Guinée-Bissau',
    'gy': 'Guyana',
    'ht': 'Haïti',
    'hn': 'Honduras',
    'hk': 'Hong Kong',
    'hu': 'Hongrie',
    'is': 'Islande',
    'in': 'Inde',
    'id': 'Indonésie',
    'ir': 'Iran',
    'iq': 'Irak',
    'ie': 'Irlande',
    'il': 'Israël',
    'it': 'Italie',
    'jm': 'Jamaïque',
    'jp': 'Japon',
    'jo': 'Jordanie',
    'kz': 'Kazakhstan',
    'ke': 'Kenya',
    'ki': 'Kiribati',
    'kp': 'Corée du Nord',
    'kr': 'Corée du Sud',
    'kw': 'Koweït',
    'kg': 'Kirghizistan',
    'la': 'Laos',
    'lv': 'Lettonie',
    'lb': 'Liban',
    'ls': 'Lesotho',
    'lr': 'Libéria',
    'ly': 'Libye',
    'li': 'Liechtenstein',
    'lt': 'Lituanie',
    'lu': 'Luxembourg',
    'mk': 'Macédoine du Nord',
    'mg': 'Madagascar',
    'mw': 'Malawi',
    'my': 'Malaisie',
    'mv': 'Maldives',
    'ml': 'Mali',
    'mt': 'Malte',
    'mh': 'Îles Marshall',
    'mr': 'Mauritanie',
    'mu': 'Maurice',
    'mx': 'Mexique',
    'fm': 'Micronésie',
    'md': 'Moldavie',
    'mc': 'Monaco',
    'mn': 'Mongolie',
    'me': 'Monténégro',
    'ma': 'Maroc',
    'mz': 'Mozambique',
    'mm': 'Myanmar',
    'na': 'Namibie',
    'nr': 'Nauru',
    'np': 'Népal',
    'nl': 'Pays-Bas',
    'nz': 'Nouvelle-Zélande',
    'ni': 'Nicaragua',
    'ne': 'Niger',
    'ng': 'Nigeria',
    'no': 'Norvège',
    'om': 'Oman',
    'pk': 'Pakistan',
    'pw': 'Palaos',
    'ps': 'Palestine',
    'pa': 'Panama',
    'pg': 'Papouasie-Nouvelle-Guinée',
    'py': 'Paraguay',
    'pe': 'Pérou',
    'ph': 'Philippines',
    'pl': 'Pologne',
    'pt': 'Portugal',
    'pr': 'Porto Rico',
    'qa': 'Qatar',
    'ro': 'Roumanie',
    'ru': 'Russie',
    'rw': 'Rwanda',
    'kn': 'Saint-Kitts-et-Nevis',
    'lc': 'Sainte-Lucie',
    'vc': 'Saint-Vincent-et-les-Grenadines',
    'ws': 'Samoa',
    'sm': 'Saint-Marin',
    'st': 'Sao Tomé-et-Principe',
    'sa': 'Arabie saoudite',
    'sn': 'Sénégal',
    'rs': 'Serbie',
    'sc': 'Seychelles',
    'sl': 'Sierra Leone',
    'sg': 'Singapour',
    'sk': 'Slovaquie',
    'si': 'Slovénie',
    'sb': 'Îles Salomon',
    'so': 'Somalie',
    'za': 'Afrique du Sud',
    'es': 'Espagne',
    'lk': 'Sri Lanka',
    'sd': 'Soudan',
    'ss': 'Soudan du Sud',
    'sr': 'Suriname',
    'sz': 'Eswatini',
    'se': 'Suède',
    'ch': 'Suisse',
    'sy': 'Syrie',
    'tw': 'Taïwan',
    'tj': 'Tadjikistan',
    'tz': 'Tanzanie',
    'th': 'Thaïlande',
    'tl': 'Timor oriental',
    'tg': 'Togo',
    'to': 'Tonga',
    'tt': 'Trinité-et-Tobago',
    'tn': 'Tunisie',
    'tr': 'Turquie',
    'tm': 'Turkménistan',
    'tv': 'Tuvalu',
    'ug': 'Ouganda',
    'ua': 'Ukraine',
    'ae': 'Émirats arabes unis',
    'gb': 'Royaume-Uni',
    'us': 'États-Unis',
    'uy': 'Uruguay',
    'uz': 'Ouzbékistan',
    'vu': 'Vanuatu',
    've': 'Venezuela',
    'vn': 'Vietnam',
    'vi': 'Îles Vierges américaines',
    'ye': 'Yémen',
    'zm': 'Zambie',
    'zw': 'Zimbabwe'}

def scrape_athlete_medals(url):
    medals, soup = [], BeautifulSoup(requests.get(url).text, 'html.parser')
    for div in soup.select('div.medaille.visible'):
        medal = {'type': 'Inconnu', 'sport': '', 'date': '', 'location': ''}
        type_map = {'1': 'Or', '2': 'Argent', '3': 'Bronze'}
        m_type = div.select_one('div.the-medal')
        if m_type:
            medal['type'] = type_map.get(m_type.get('data-medal'), 'Inconnu')
        sport = div.select_one('div.m-sport')
        date = div.select_one('div.m-event-am')
        location = div.select_one('div.m-event-stadt')
        medal['sport'] = sport.text.strip() if sport else ''
        medal['date'] = date.text.strip()[3:] if date else ''
        medal['location'] = location.text.strip()[3:] if location else ''
        medals.append(medal)
    return medals

def scrape_athletes_by_letter(url):
    athletes, soup = [], BeautifulSoup(requests.get(url).text, 'html.parser')
    for card in soup.select('a.card.athlet'):
        url_ = card.get('href', '')
        if not url_.startswith('http'):
            url_ = f"https://olympics-statistics.com{url_}"
        fn = card.select_one('div.vn')
        ln = card.select_one('div.nn div.n')
        flag = card.select_one('img[src*="flagge"]')
        gender_icon = card.select_one('svg[class*="male"]')

        # Nom
        first = fn.text.strip() if fn else ''
        last = ln.text.strip() if ln else ''

        # Pays
        code = re.search(r'/flagge[s]?/([a-z]{2,5})\.png', flag['src']) if flag else None
        country = COUNTRY_CODES.get(code.group(1).lower(), code.group(1).upper()) if code else ''

        # Genre
        classes = ' '.join(gender_icon.get('class', [])) if gender_icon else ''
        gender = 'Femme' if 'female' in classes else 'Homme' if 'male' in classes else ''

        athletes.append({
            'firstName': first,
            'lastName': last,
            'url': url_,
            'country': country,
            'gender': gender,
            'medals': scrape_athlete_medals(url_)
        })

        print(f"{first} {last} ({country})")

    return athletes

def main():
    all_athletes = []
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ#"
    for letter in alphabet:
        print(f"Lettre {letter}")
        url = f"https://olympics-statistics.com/olympic-athletes/{letter.lower() if letter != '#' else letter}"
        athletes = scrape_athletes_by_letter(url)
        all_athletes.extend(athletes)
        with open("olympic_athletes.json", "w", encoding="utf-8") as f:
            json.dump(all_athletes, f, indent=2, ensure_ascii=False)
        print(f"Total cumulé : {len(all_athletes)}")

if __name__ == "__main__":
    main()


In [18]:
import requests, json
from bs4 import BeautifulSoup

def get_nation_medals(url):
    medals = {'gold': 0, 'silver': 0, 'bronze': 0, 'total': 0}
    soup = BeautifulSoup(requests.get(url).text, 'html.parser')
    for code, name in [('1', 'gold'), ('2', 'silver'), ('3', 'bronze')]:
        div = soup.select_one(f'div.the-medal[data-medal="{code}"]')
        if div:
            span = div.parent.select_one('span.mal')
            if span and span.text.strip().isdigit():
                medals[name] = int(span.text.strip())
    medals['total'] = sum([medals[k] for k in ['gold', 'silver', 'bronze']])
    return medals

def scrape_nations(max_nations=178):
    url = "https://olympics-statistics.com/nations"
    soup = BeautifulSoup(requests.get(url).text, 'html.parser')
    cards = soup.select('a.card.nation.visible')[:max_nations]
    nations = []

    for i, card in enumerate(cards):
        href = card.get('href', '')
        full_url = href if href.startswith('http') else f"https://olympics-statistics.com{href}"
        name_tag = card.select_one('div.bez')
        name = name_tag.text.strip() if name_tag else "Unknown"
        print(f"{i+1}/{max_nations} - {name}")
        medals = get_nation_medals(full_url)
        nations.append({'name': name, 'medals': medals})
    
    return nations

def main():
    data = scrape_nations()
    with open("olympic_nations_medals.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("Données enregistrées dans olympic_nations_medal.json")

if __name__ == "__main__":
    main()


1/178 - Afghanistan
2/178 - Albania
3/178 - Algeria
4/178 - Argentina
5/178 - Armenia
6/178 - Australia
7/178 - Australia / New Zealand (Australasia)
8/178 - Austria
9/178 - Austria / United States of Amerika
10/178 - Azerbaijan
11/178 - Bahamas
12/178 - Bahrain
13/178 - Barbados
14/178 - Belarus
15/178 - Belgium
16/178 - Bermuda
17/178 - Bohemia
18/178 - Bohemia / Great Britain
19/178 - Botswana
20/178 - Brazil
21/178 - Bulgaria
22/178 - Burkina-Faso
23/178 - Burundi
24/178 - Cameroon
25/178 - Canada
26/178 - Cape Verde
27/178 - Chile
28/178 - China
29/178 - Chinese Taipei
30/178 - Columbia
31/178 - Commonwealth of Independent States
32/178 - Costa Rica
33/178 - Cote d´Ivoire
34/178 - Croatia
35/178 - Cuba
36/178 - Cyprus
37/178 - Czech Republic
38/178 - Czechoslovakia
39/178 - Denmark
40/178 - Djibouti
41/178 - Dominican
42/178 - Dominican Republic
43/178 - Ecuador
44/178 - Egypt
45/178 - Eritrea
46/178 - Estonia
47/178 - Ethiopia
48/178 - Federal Republic of Germany
49/178 - Fiji
50

In [6]:
import requests, json, re
from bs4 import BeautifulSoup

def scrape_sports_medals_by_nation(max_sports=68):
    url = "https://olympics-statistics.com/olympic-sports"
    all_sports = []

    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')

    sports = []
    for a in soup.select('a'):
        href = a.get('href', '')
        if '/olympic-sport/' in href:
            name = a.text.strip() or re.search(r'/olympic-sport/([^/]+)', href).group(1).replace('-', ' ').title()
            full_url = href if href.startswith('http') else f"https://olympics-statistics.com{href}"
            if full_url not in [s['url'] for s in sports]:
                sports.append({'name': name, 'url': full_url})

    print(f"{len(sports)} sports valides trouvés")

    for i, sport in enumerate(sports[:max_sports]):
        print(f"{i+1}/{max_sports} → {sport['name']}")
        r = requests.get(sport['url'])
        soup_sport = BeautifulSoup(r.text, 'html.parser')
        cards = soup_sport.select('div.card.nation, a.card.nation, [class*="nation"]')
        nations, pos = [], 1

        for card in cards:
            name_tag = card.select_one('div.bez')
            if not name_tag: continue
            name = name_tag.text.strip().split('\n')[0].strip()
            spans = card.select('span.mal')
            if len(spans) >= 3:
                try:
                    g = int(spans[0].text.strip())
                    s = int(spans[1].text.strip())
                    b = int(spans[2].text.strip())
                except:
                    continue
                total = g + s + b
                if total:
                    nations.append({'position': pos, 'name': name, 'medals': {'gold': g, 'silver': s, 'bronze': b, 'total': total}})
                    pos += 1

        if nations:
            all_sports.append({**sport, 'nations': nations})

    return all_sports

data = scrape_sports_medals_by_nation()
with open("olympic_sports_medals.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print("Données enregistrées.")


68 sports valides trouvés
1/68 → Aeronautics
2/68 → Alpine Skiing
3/68 → Alpinism
4/68 → Archery
5/68 → Athletics
6/68 → Badminton
7/68 → Baseball
8/68 → Basketball
9/68 → Beachvolleyball
10/68 → Biathlon
11/68 → Bobsledding
12/68 → Boxing
13/68 → Breaking
14/68 → Canoe & Kayaking
15/68 → Cricket
16/68 → Crocket
17/68 → Cross-Country Skiing
18/68 → Curling
19/68 → Cycling
20/68 → Diving
21/68 → Equestrian
22/68 → Fencing
23/68 → Figure Skating
24/68 → Football
25/68 → Freestyle Skiing
26/68 → Golf
27/68 → Gymnastics
28/68 → Handball
29/68 → Hockey
30/68 → Ice Hockey
31/68 → Jeu de Paume
32/68 → Judo
33/68 → Karate
34/68 → Lacrosse
35/68 → Luge
36/68 → Modern Pentathlon
37/68 → Motorboating
38/68 → Nordic Combined
39/68 → Pelota
40/68 → Polo
41/68 → Rackets
42/68 → Rhythmic Gymnastics
43/68 → Roque
44/68 → Rowing
45/68 → Rugby
46/68 → Sailing
47/68 → Shooting
48/68 → Short Track
49/68 → Skateboarding
50/68 → Skeleton
51/68 → Ski Jumping
52/68 → Snowboard
53/68 → Softball
54/68 → Speed S